# 4주차 정답 Notebook — 결측값과 이상한 데이터 정리하기

## 1단계. Pandas 불러오고 데이터 읽기

In [1]:
import pandas as pd
df = pd.read_csv("../../data/weekly/week04/week04_dirty_process_data.csv")
df.head()

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
0,2024-03-01 23:00,LOT-0336,EQ-01,식각,297.5,1010.0,49.92,123.1,1.0
1,2024-03-02 08:00,LOT-0281,EQ-02,포토,299.3,1012.1,47.91,114.6,1.0
2,2024-03-02 12:00,LOT-0304,EQ-01,세정,297.5,1008.9,51.20,115.3,1.0
3,2024-03-02 16:00,LOT-0254,EQ-02,식각,294.8,1024.5,51.29,110.5,-1.0
4,2024-03-03 02:00,LOT-0020,EQ-04,산화,302.6,997.3,45.75,117.4,1.0


## 2단계. 데이터 크기 확인하기

In [2]:
print(df.shape)

(113, 9)


**질문 답**: 이 데이터는 113행 9열로 이루어져 있다.

## 3단계. 결측값 개수 세기

In [3]:
print(df.isna().sum())

측정시간        0
로트번호        0
설비번호        0
공정명         0
온도_섭씨       4
압력_Pa       3
가스유량_slm    1
처리시간_sec    0
합격여부        2
dtype: int64


**질문 답**: `합격여부`가 아니라 `온도_섭씨` 열에 결측값이 4개로 가장 많다.

## 4단계. 결측값이 있는 행 직접 보기

In [4]:
df[df.isna().any(axis=1)]

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
6,2024-03-04 17:00,LOT-0287,EQ-01,증착,298.6,1021.4,NaN,131.8,1.0
25,2024-03-10 05:00,LOT-0168,EQ-04,중착,NaN,1006.2,53.36,115.0,1.0
29,2024-03-11 19:00,LOT-0180,EQ-02,세정,295.5,NaN,49.99,117.6,1.0
34,2024-03-13 12:00,LOT-0396,EQ-04,산화,298.7,NaN,54.86,113.7,1.0
44,2024-03-17 05:00,LOT-0232,EQ-02,세정,303.3,1006.2,49.12,117.1,NaN
76,2024-03-26 07:00,LOT-0039,EQ-02,산화,NaN,1022.4,45.96,112.8,1.0
83,2024-03-28 13:00,LOT-0219,EQ-04,식각,303.6,NaN,49.71,119.7,NaN
96,2024-04-03 12:00,LOT-0201,EQ-01,산화,NaN,1007.4,50.89,124.6,1.0
106,2024/03/05,LOT-0383,EQ-04,증착,NaN,1011.0,47.50,116.6,1.0


## 5단계. 결측값 제거 vs 대체 비교

In [5]:
df_dropped = df.dropna()
print("제거 후 행 개수:", df_dropped.shape[0])

제거 후 행 개수: 104


In [6]:
median_temp = df["온도_섭씨"].median()
print("온도_섭씨 중앙값:", median_temp)

온도_섭씨 중앙값: 299.3


**질문 답**: 대체(`fillna`) 방법이 행 개수를 더 많이 유지한다(제거는 결측이 있는 행을 통째로 없애기 때문).

## 6단계. 중복 행 찾고 정리하기

In [7]:
print(df.duplicated().sum())

df_no_dup = df.drop_duplicates()
print("중복 제거 후 행 개수:", df_no_dup.shape[0])

3
중복 제거 후 행 개수: 110


## 7단계. 이상값 찾기

In [8]:
df[df["압력_Pa"] < 0]

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
37,2024-03-13 16:00,LOT-0238,EQ-03,식각,308.4,-40.370466,50.85,126.0,1.0
79,2024-03-26 21:00,LOT-0121,EQ-04,산화,300.9,-28.297835,44.92,113.5,1.0


In [9]:
df[df["온도_섭씨"] > 400]

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
17,2024-03-07 23:00,LOT-0202,EQ-04,포토,520.879883,1017.4,51.59,119.6,1.0
67,2024-03-24 08:00,LOT-0177,EQ-01,증착,519.854978,1029.9,48.75,123.7,1.0


## 8단계(도전). 정제 전후 비교하기

In [10]:
df_clean = df.drop_duplicates().copy()

bad_temp = df_clean["온도_섭씨"] > 400
bad_pressure = df_clean["압력_Pa"] < 0
bad_gas = (df_clean["가스유량_slm"] < 0) | (df_clean["가스유량_slm"] > 200)
df_clean = df_clean[~(bad_temp | bad_pressure | bad_gas)].copy()

df_clean = df_clean.dropna(subset=["합격여부"])

df_clean = df_clean.fillna({
    "온도_섭씨": df_clean["온도_섭씨"].median(),
    "압력_Pa": df_clean["압력_Pa"].median(),
    "가스유량_slm": df_clean["가스유량_slm"].median(),
})

print("정제 후 행 개수:", df_clean.shape[0])
print("정제 후 결측값 합계:", df_clean.isna().sum().sum())

정제 후 행 개수: 102
정제 후 결측값 합계: 0


In [11]:
mean_before = df["온도_섭씨"].mean()
mean_after = df_clean["온도_섭씨"].mean()
print("정제 전 평균 온도:", round(mean_before, 2))
print("정제 후 평균 온도:", round(mean_after, 2))
print("행 개수:", df.shape[0], "->", df_clean.shape[0])

정제 전 평균 온도: 303.86
정제 후 평균 온도: 299.69
행 개수: 113 -> 102


**질문 답**: 정제 전 평균 약 303.86℃, 정제 후 평균 약 299.69℃, 차이는 약 4.18℃이다. 그 이유는 정제 전 데이터에 500℃가 넘는 극단 고온 이상값 2건이 섞여 있어 평균을 끌어올렸기 때문이다.

## 9단계. 오늘의 분석을 한 문장으로 정리하기

> 원본 데이터 113행 중 중복 3건, 이상값(극단 고온·음수 압력·비정상 가스유량) 6건, 합격여부 결측 2건을 정리한 결과 최종 102행이 남았으며, 정제 전후 평균 온도는 약 4.18℃ 차이가 났다. 정제 전 평균이 정상 범위 상단에 걸려 보였던 것은 실제 공정 이상이 아니라 이상값 2건 때문이었다.